# U6 reliability threshold calibration

## Observed outcome

At the starting thresholds, 0 of 60 nominally insufficient trials were admitted (0.0%). All 20 full-coverage trials were admitted; only 4 of 20 trials at nominal 20% coverage were admitted. These finite synthetic results do not establish a field error rate.

## Context and method

This seeded experiment audits admission to the U6 reliability gate at 20% employer coverage and 30 effective moves per direction. It does not test whether an agreement is flagged. All six populations are null markets generated locally by U2; there are no planted agreements.

### Key assumptions

Coverage is year-end distinct workers divided by exact synthetic truth, for every year 2015–2024. Nominal negatives have an employer sampling probability below 20% or fewer than 30 true distinct directional movers. False admissions are nominal negatives declared sufficient. A 19% sampling probability can realize more than 20% coverage by chance: this rate measures nominal-population admission, not incorrect arithmetic on realized coverage.

In this two-employer null, effective samples are unique workers with both move endpoints observed; no weights are needed and repeated workers count once. This fixture proxy does not define U7's effective opportunity sample for suppressed-flow tests. Twenty seeds per case characterize the starting thresholds; they do not select an optimal threshold or validate real-data behavior. The aggregate is a case-mix-dependent diagnostic, not an estimated field error rate.

Sources: `src/nopoach/synth/generator.py`, `src/nopoach/synth/scenarios.py`; thresholds and gate: `src/nopoach/market/sufficiency.py`. No external data or network. Run from the repository root with `PYTHONPATH=src`; the pytest notebook check executes cells top to bottom without Jupyter dependencies.


## Data and parameters


In [1]:
from collections import defaultdict
from dataclasses import replace

from nopoach.market.coverage import (
    Headcount,
    MarketWorker,
    coverage_report,
    estimate_workforce,
)
from nopoach.market.define import Employer, Geography, Market
from nopoach.market.sufficiency import (
    MIN_COVERAGE,
    MIN_EFFECTIVE_MOVES,
    signal_sufficiency,
)
from nopoach.synth.generator import generate_market
from nopoach.synth.scenarios import CoverageMask, named_scenario

SEEDS = range(20)
# Nominal sampling fractions, with a low-movement null as a sample-size control.
CASES = [
    ("thin_8_percent", 0.08, 0.04),
    ("near_19_percent", 0.19, 0.04),
    ("boundary_20_percent", 0.20, 0.04),
    ("half_coverage", 0.50, 0.04),
    ("full_coverage", 1.00, 0.04),
    ("too_few_moves", 1.00, 0.001),
]
print(f"Thresholds: coverage={MIN_COVERAGE:.0%}, effective moves={MIN_EFFECTIVE_MOVES}")

Thresholds: coverage=20%, effective moves=30


## Calculation


In [2]:
def year_end_rows(rows, year):
    cutoff = f"{year}-12-31"
    return [
        r
        for r in rows
        if r["start_date"] <= cutoff and (not r["end_date"] or r["end_date"] >= cutoff)
    ]


def unique_directional_movers(generated, observed):
    visible_ends = {
        (r["person_key"], r["employer_id"], r["end_date"])
        for r in observed
        if r["end_date"]
    }
    visible_starts = {
        (r["person_key"], r["employer_id"], r["start_date"]) for r in observed
    }
    people = defaultdict(set)
    for move in generated.moves:
        if (
            move.person_key,
            move.source,
            f"{move.year - 1}-12-31",
        ) in visible_ends and (
            move.person_key,
            move.destination,
            f"{move.year}-01-01",
        ) in visible_starts:
            people[move.source, move.destination].add(move.person_key)
    keys = [e.key for e in generated.scenario.employers]
    return {(a, b): len(people[a, b]) for a in keys for b in keys if a != b}


results = []
for label, fraction, move_rate in CASES:
    admitted = false_admissions = nominal_negatives = 0
    for seed in SEEDS:
        base = named_scenario("AE3")
        scenario = replace(
            base,
            name=f"calibration_{label}",
            coverage=(CoverageMask("processor-a", fraction),),
            occupations=(replace(base.occupations[0], yearly_move_rate=move_rate),),
        )
        generated = generate_market(scenario, seed=seed)
        market = Market(frozenset({"51-3022"}), Geography("state", "MN"), 2015, 2024)
        employers = [Employer(e.key, e.name) for e in scenario.employers]
        estimates, workers = [], []
        for year in range(market.start_year, market.end_year + 1):
            truth = year_end_rows(generated.true_rows, year)
            for employer in employers:
                count = len(
                    {r["person_key"] for r in truth if r["employer_id"] == employer.id}
                )
                estimates.append(
                    estimate_workforce(
                        employer.id,
                        year,
                        market,
                        imported=Headcount(
                            count, count, count, "U2 synthetic year-end truth"
                        ),
                    )
                )
            workers.extend(
                MarketWorker(
                    r["person_key"],
                    r["employer_id"],
                    r["occupation_code"],
                    market.geography,
                    year,
                )
                for r in year_end_rows(generated.rows, year)
            )
        samples = unique_directional_movers(generated, generated.rows)
        true_samples = unique_directional_movers(generated, generated.true_rows)
        # Independent nominal population target, rather than recomputing the gate.
        nominally_insufficient = fraction < 0.20 or min(true_samples.values()) < 30
        verdict = signal_sufficiency(
            coverage_report(market, employers, estimates, workers),
            samples,
            clause_feeds=[],
            tie_feeds=[],
        )
        reliable = verdict.flow.status == "sufficient"
        admitted += int(reliable)
        nominal_negatives += int(nominally_insufficient)
        false_admissions += int(reliable and nominally_insufficient)
    results.append(
        {
            "case": label,
            "trials": len(SEEDS),
            "admitted": admitted,
            "nominal_negatives": nominal_negatives,
            "false_admissions": false_admissions,
            "false_admission_rate": false_admissions / nominal_negatives
            if nominal_negatives
            else None,
        }
    )

## Results

Rate is false admissions / nominal negatives; n/a means no nominal negatives.


In [3]:
print(
    "Case                 Trials  Admitted  Nominal negatives  False admissions  Rate"
)
for row in results:
    rate = (
        "n/a"
        if row["false_admission_rate"] is None
        else f"{row['false_admission_rate']:.1%}"
    )
    print(
        f"{row['case']:<20} {row['trials']:>6} {row['admitted']:>9} "
        f"{row['nominal_negatives']:>18} {row['false_admissions']:>17} {rate:>6}"
    )
negatives = sum(row["nominal_negatives"] for row in results)
false_admissions = sum(row["false_admissions"] for row in results)
print(
    f"Nominal reliability-gate false-positive rate: {false_admissions}/{negatives} "
    f"= {false_admissions / negatives:.1%}"
)
print(
    "Agreement-screening false-positive rate: unavailable until U7 "
    "flow/placebo/BH implementation."
)

Case                 Trials  Admitted  Nominal negatives  False admissions  Rate
thin_8_percent           20         0                 20                 0   0.0%
near_19_percent          20         0                 20                 0   0.0%
boundary_20_percent      20         4                  0                 0    n/a
half_coverage            20        20                  0                 0    n/a
full_coverage            20        20                  0                 0    n/a
too_few_moves            20         0                 20                 0   0.0%
Nominal reliability-gate false-positive rate: 0/60 = 0.0%
Agreement-screening false-positive rate: unavailable until U7 flow/placebo/BH implementation.


## Interpretation

Keep the specified starting thresholds provisional. Boundary sampling variation and uncertainty in real workforce denominators remain material; the exact synthetic denominators here do not measure those errors. SEC/BLS allocation bands are sensitivity bands, not fitted intervals. U7 must supply flow statistics, empirical-Bayes shrinkage and placebo/BH results before an agreement-screening false-positive rate can be calibrated. No statistical signal was added to U6.
